# Netflix Data Cleaning and EDA

This notebook cleans the raw Netflix titles dataset, engineers a few useful features, and performs a small exploratory analysis. The output is saved to `data/processed/netflix_clean.csv` so it can power the Streamlit dashboard.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid')

## Load data

In [ ]:
raw_path = Path('../data/raw/netflix_titles.csv')
df = pd.read_csv(raw_path)
df.head()

## Initial audit

In [ ]:
print('Shape:', df.shape)
df.info()
df.isna().sum().sort_values(ascending=False).head(12)

## Cleaning and feature engineering

In [ ]:
clean = df.copy()

# Standardize column names
clean.columns = [c.strip().lower().replace(' ', '_') for c in clean.columns]

# Parse dates
clean['date_added'] = pd.to_datetime(clean['date_added'], errors='coerce')
clean['year_added'] = clean['date_added'].dt.year
clean['month_added'] = clean['date_added'].dt.month_name()

# Clean text fields
for col in ['director', 'cast', 'country', 'rating', 'listed_in']:
    if col in clean.columns:
        clean[col] = clean[col].fillna('Unknown').astype(str).str.strip()

# Split duration into numeric value and unit
if 'duration' in clean.columns:
    duration_split = clean['duration'].astype(str).str.extract(r'(?P<duration_value>\d+)\s*(?P<duration_unit>\w+)')
    clean['duration_value'] = pd.to_numeric(duration_split['duration_value'], errors='coerce')
    clean['duration_unit'] = duration_split['duration_unit'].str.lower()

# Remove exact duplicates
clean = clean.drop_duplicates()

# Create helper features
clean['is_movie'] = clean['type'].eq('Movie')
clean['is_tv_show'] = clean['type'].eq('TV Show')

clean.head()

## Save cleaned data

In [ ]:
processed_path = Path('../data/processed/netflix_clean.csv')
processed_path.parent.mkdir(parents=True, exist_ok=True)
clean.to_csv(processed_path, index=False)
processed_path

## Basic exploratory analysis

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
clean['type'].value_counts().plot(kind='bar', ax=ax)
ax.set_title('Movies vs TV Shows')
ax.set_xlabel('')
ax.set_ylabel('Count')
plt.tight_layout()

In [ ]:
yearly = clean.groupby('year_added', dropna=True).size().reset_index(name='count')
yearly = yearly.sort_values('year_added')
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(yearly['year_added'], yearly['count'], marker='o')
ax.set_title('Titles Added Over Time')
ax.set_xlabel('Year Added')
ax.set_ylabel('Titles')
plt.tight_layout()

In [ ]:
genres = clean['listed_in'].fillna('Unknown').str.split(',').explode().str.strip()
top_genres = genres.value_counts().head(10)
fig, ax = plt.subplots(figsize=(8, 4))
top_genres.sort_values().plot(kind='barh', ax=ax)
ax.set_title('Top 10 Genres')
ax.set_xlabel('Count')
ax.set_ylabel('')
plt.tight_layout()

## Summary insights

In [ ]:
summary = pd.DataFrame({
    'metric': ['total_titles', 'movies', 'tv_shows', 'countries'],
    'value': [
        len(clean),
        int(clean['type'].eq('Movie').sum()),
        int(clean['type'].eq('TV Show').sum()),
        int(clean['country'].nunique()),
    ],
})
summary

### Interpretation
- The dataset is primarily useful for content mix and trend analysis.
- Cleaning focuses on date parsing, duplicate removal, and splitting compound fields.
- The processed file is designed for the Streamlit dashboard.